# Chapter 2 — Geometric memory in one chapter

The GMS store is a **trained triple register**, not a vector database.
This notebook exercises the six primitives at agent-author altitude:
`lookup_enm`, `score_triple`, `query_triples`, `link_predict`,
`check_holonomy`, `tension_energy` — over the Northwind Industries FY2025
store built by `scripts/build_store.py`. Geometry internals (rotors, caps,
transport) are deferred to Appendix C and the GMS monograph.

The claim this notebook proves: **exact figures come back byte-exact from
Exact Numerical Memory — never re-parsed from prose — and a false fact sits
measurably farther from the manifold than the true one.**

In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/home/asudjianto/jupyterlab/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)

## Load the trained store

A store is reopened with the *same* geometry/loss configuration it was
trained under (`scripts/build_store.py`): cap-admissibility loss and a
64/64/32/32 geometry. `load()` rebuilds the model, the entity/relation
adapter, the document graph, and the Exact Numerical Memory from disk.

In [ ]:
import torch
from knowlytix.core.config import GeometryConfig
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore

REPO_ROOT = os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
STORE = os.path.join(REPO_ROOT, "data", "gms_annual_report_store")

cfg = DocGMSConfig(
    store_path=STORE,
    ingest_mode="regex",
    loss_mode="cap",
    geometry=GeometryConfig(d_v=64, d_u=64, m=32, d=32),
)
store = GMSExpertStore(cfg, device=torch.device("cpu"))
assert store.load(), f"no store at {STORE} — run scripts/build_store.py first"
print("relations:", sorted(store.adapter.relation_to_idx))

## Primitive 1 — `lookup_enm`: exact recall vs re-parsing prose

Every authoritative number lives in **Exact Numerical Memory** under a
`(category, id)` key with SHA-256 integrity. `lookup_enm` returns the value
byte-exact. The chunk-and-pray alternative — regexing a figure out of
retrieved text — is fragile: the same digits appear in multiple sentences,
thousands separators and currency glyphs vary, and a near-miss looks like a
hit. Below we read FY2025 revenue from ENM, then deliberately break a naive
parse of the same number from prose.

In [ ]:
import re

# Authoritative path: byte-exact read from Exact Numerical Memory.
revenue_fy2025 = store.lookup_enm("income_statement", "Revenue/FY2025")
total_segment_rev = store.lookup_enm("segment_performance", "Total/All/Revenue")
print("ENM income_statement/Revenue/FY2025 =", revenue_fy2025)
print("ENM segment_performance/Total/All/Revenue =", total_segment_rev)

# Chunk-and-pray path: parse a figure out of a retrieved sentence.
# Two sentences mention '355' — and one mentions the WRONG prior-year basis.
retrieved_prose = (
    "Total revenue grew to $355M in FY2025, up from $320M, while the "
    "Cloud Platform segment alone contributed $120M."
)
first_number = float(re.search(r"\$(\d+)M", retrieved_prose).group(1))
print("naive parse (first $NM match) =", first_number)

# The parse is right here only by luck of word order. Reorder the clause and
# the same regex now returns the SEGMENT figure, not total revenue:
reordered = (
    "The Cloud Platform segment contributed $120M as total revenue "
    "grew to $355M in FY2025."
)
wrong = float(re.search(r"\$(\d+)M", reordered).group(1))
print("naive parse after reorder =", wrong, "(should be 355, got", wrong, ")")
assert wrong != revenue_fy2025, "the prose parse silently returned the wrong number"
print("ENM is order-invariant and exact; the parse is neither.")

## Primitive 2 — `score_triple`: a true fact vs a false one

`score_triple(head, rel, tail)` returns a geodesic distance on the trained
manifold — **lower is more plausible**. A fact the store was trained on sits
close to the conditioned cap center; a fabricated tail sits farther away. We
read the gap between the true Cloud Platform headcount (340) and a fabricated
one. Tails are canonicalized numeric strings (`"340.0"`), matching the
triples in `data/corpus_facts.md`.

In [ ]:
true_d = store.score_triple("cloud platform", "has_headcount", "340.0")
false_d = store.score_triple("cloud platform", "has_headcount", "520.0")  # that's retail's
print(f"d(cloud platform, has_headcount, 340.0) = {true_d:.4f}  [asserted]")
print(f"d(cloud platform, has_headcount, 520.0) = {false_d:.4f}  [false]")
print(f"geodesic gap (false - true) = {false_d - true_d:.4f}")

# The asserted fact scores strictly closer than the swapped tail.
assert true_d < false_d, "true fact must score closer than the false one"
rho = store.cap_radius("has_headcount")
print("cap radius rho for has_headcount =", rho)

## Primitive 3 — `query_triples`: asserted edges over a segment

`query_triples` is exact pattern-matching over the document graph — no
geometry, no guessing. Leave a slot `None` to wildcard it. This is the path
the retriever prefers: asserted edges with provenance, never link-prediction
guesses.

In [ ]:
cloud_edges = store.query_triples(head="cloud platform")
for h, r, t in cloud_edges:
    print(f"{h:>16} {r:>14} {t}")

# All four segments that report a revenue edge:
rev_edges = store.query_triples(relation="has_revenue")
segments = {h: t for (h, r, t) in rev_edges if h != "total"}
print("\nper-segment revenue:", segments)
assert ("cloud platform", "has_division", "technology") in cloud_edges

## Primitive 4 — `link_predict`: ranked tails for an open slot

When you have `(head, relation, ?)` and want the store's *ranked* guess,
`link_predict` scores every type-valid tail and returns the closest ones
(lower distance first). It is type-constrained: only entities ever seen as a
tail of this relation are scored, so numeric and categorical tails never mix.
Note this is a **prediction**, not an assertion — Chapter 8 shows why the
retriever prefers an asserted `query_triples` edge over a `link_predict`
guess whenever one exists.

In [ ]:
ranked = store.link_predict("cloud platform", "has_division", top_k=3)
for tail, dist in ranked:
    print(f"{tail:>14}  d={dist:.4f}")

# Cloud Platform's division is Technology — it should rank first.
assert ranked, "link_predict returned no candidates"
top_tail, _ = ranked[0]
print("top-ranked division:", top_tail)

## Primitive 5 — `check_holonomy`: is a multi-hop path consistent?

`check_holonomy(path, direct)` measures the *holonomy defect* — how far
composing a relation path drifts from a direct edge. **0 = consistent.** It
is the geometric backbone of multi-hop retrieval (Chapter 8) and of GEODE's
composition critic (Chapter 5). Here we ask whether composing
`has_division` then `has_region` is consistent with a hypothetical direct
`has_region` edge from a segment. The store has no direct segment→region
edge, so we read the defect of the path against the `has_region` rotor.

In [ ]:
defect = store.check_holonomy(["has_division", "has_region"], "has_region")
print("holonomy defect for has_division ∘ has_region vs has_region =", defect)
tau = store.config.verify.tau_path
print("path-consistency threshold tau_path =", tau)
# A defect at or below tau_path is treated as a consistent composition.
assert defect is not None, "both relations must exist for a holonomy read"

## Primitive 6 — `tension_energy`: contradiction as distance

`tension_energy(a, b)` reads the relationship between two entities on a
0..2 scale: **0 = agree, √2 ≈ 1.41 = unrelated, 2 = contradict.** Functional
facts (one division head, one fiscal-year-end) make contradiction a
*geometric* signal rather than a string compare. Below: the two real
division heads (`dana cole`, `sam reyes`) are distinct people running
distinct divisions — the energy reads them as not-agreeing, which is exactly
what flags a 'two CEOs' style contradiction in Chapter 5 and Chapter 10.

In [ ]:
te_heads = store.tension_energy("dana cole", "sam reyes")
te_self = store.tension_energy("dana cole", "dana cole")
print(f"tension(dana cole, sam reyes) = {te_heads:.4f}  [two distinct heads]")
print(f"tension(dana cole, dana cole) = {te_self:.4f}  [identical -> agree]")
assert te_self <= te_heads, "an entity must not contradict itself"
print("contradiction is a distance, not a string mismatch.")

## Self-check — the chapter's claim

ENM recall is byte-exact, and the asserted fact scores strictly closer than
the false one. If this cell passes, the store behaved as a trained register,
not a fuzzy index.

In [ ]:
# 1. ENM is exact: FY2025 revenue and total segment revenue both equal 355.0.
assert store.lookup_enm("income_statement", "Revenue/FY2025") == 355.0
assert store.lookup_enm("segment_performance", "Total/All/Revenue") == 355.0
assert store.lookup_enm("segment_performance", "Cloud Platform/Technology/Headcount") == 340.0

# 2. A true fact sits closer on the manifold than a fabricated one.
d_true = store.score_triple("cloud platform", "has_headcount", "340.0")
d_false = store.score_triple("cloud platform", "has_headcount", "520.0")
assert d_true < d_false

print("OK: ENM byte-exact (355.0 / 340.0) and true-fact geodesic gap =",
      f"{d_false - d_true:.4f}")